# 01 - Data Cleaning & Feature Engineering
Loads the raw Olist tables, merges them, cleans the result and builds the customer-level feature table.

**Outputs** (saved to `processed/`):
- `olist_cleaned_items.csv` - item-level cleaned data
- `customer_features.csv` - one row per customer

In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
# Mount Google Drive (Colab)
from google.colab import drive
drive.mount('/content/drive')

drive_path = '/content/drive/MyDrive/olist/'          # <-- raw Olist CSVs (update to your folder)
processed_path = drive_path + 'processed/'            # <-- intermediate files shared between notebooks
os.makedirs(processed_path, exist_ok=True)

In [ ]:
# ==============================================================================
# STEP 1: LOAD DATA (Google Drive mounted in Colab)
# ==============================================================================


print("Loading datasets from Google Drive...")
orders = pd.read_csv(drive_path + 'olist_orders_dataset.csv')
customers = pd.read_csv(drive_path + 'olist_customers_dataset.csv')
items = pd.read_csv(drive_path + 'olist_order_items_dataset.csv')
products = pd.read_csv(drive_path + 'olist_products_dataset.csv')
payments = pd.read_csv(drive_path + 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(drive_path + 'olist_order_reviews_dataset.csv')
sellers = pd.read_csv(drive_path + 'olist_sellers_dataset.csv')
category_translation = pd.read_csv(drive_path + 'product_category_name_translation.csv')

# Merge everything into one wide table, one join at a time
print("Merging core tables...")
df = pd.merge(orders, customers, on='customer_id', how='left')
df = pd.merge(df, items, on='order_id', how='left')

print("Adding product details...")
df = pd.merge(df, products, on='product_id', how='left')
df = pd.merge(df, category_translation, on='product_category_name', how='left')

print("Adding payments, reviews, and sellers...")
df = pd.merge(df, payments, on='order_id', how='left')
df = pd.merge(df, reviews, on='order_id', how='left')
df = pd.merge(df, sellers, on='seller_id', how='left')

print(f"Dataset created successfully! Rows: {df.shape[0]}  Columns: {df.shape[1]}")
display(df.head(3))



In [ ]:
# ==============================================================================
# STEP 2: CLEAN & ENGINEER FEATURES
# ==============================================================================

# 1. Keep only completed orders
full = df[df['order_status'] == 'delivered'].copy()

# 2. Convert date strings to actual datetime objects
date_columns = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_columns:
    full[col] = pd.to_datetime(full[col])

# 3. Drop rows missing critical identifiers or pricing
full.dropna(subset=['customer_unique_id', 'price'], inplace=True)

# 4. Remove duplicates caused by the multi-table merge (e.g. multiple payments per order)
#    Keep unique combinations of order_id and order_item_id -- one row per item purchased
full = full.drop_duplicates(subset=['order_id', 'order_item_id']).copy()

print(f"Cleaned item-level dataset ready! Rows: {full.shape[0]}")

In [ ]:
# Feature engineering on top of the cleaned table
full["delivery_delay_days"] = (full["order_delivered_customer_date"] - full["order_estimated_delivery_date"]).dt.days
full["is_late"] = full["delivery_delay_days"] > 0
full["total_order_value"] = full["price"] + full["freight_value"]

# Customer-level table (this is what feeds RFM, clustering, and the churn model)
ref_date = full["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

customer_features = full.groupby("customer_unique_id").agg(
    frequency=("order_id", "nunique"),
    monetary_value=("total_order_value", "sum"),
    last_order=("order_purchase_timestamp", "max"),
    category_count=("product_category_name_english", "nunique"),
    average_review_score=("review_score", "mean"),
    late_order_rate=("is_late", "mean"),
    average_freight_cost=("freight_value", "mean"),
    average_installments=("payment_installments", "mean"),
).reset_index()

customer_features["recency_days"] = (ref_date - customer_features["last_order"]).dt.days
customer_features["average_order_value"] = customer_features["monetary_value"] / customer_features["frequency"]

print(f"customer_features: {customer_features.shape[0]} customers, {customer_features.shape[1]} columns")

In [ ]:
# ==============================================================================
# SAVE CLEANED DATA for the EDA / segmentation and model notebooks
# ==============================================================================
full.to_csv(processed_path + 'olist_cleaned_items.csv', index=False)
customer_features.to_csv(processed_path + 'customer_features.csv', index=False)
print(f"Saved olist_cleaned_items.csv     ({full.shape[0]} rows)")
print(f"Saved customer_features.csv       ({customer_features.shape[0]} rows)")